In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from collections import defaultdict
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
from dysts.metrics import compute_metrics
from tqdm import tqdm

from panda.chronos.pipeline import ChronosPipeline
from panda.patchtst.pipeline import PatchTSTPipeline
from panda.utils import (
    DEFAULT_MARKERS,
    apply_custom_style,
    safe_standardize,
)

apply_custom_style("../config/plotting.yaml")

In [ ]:
WORK = os.environ.get("WORK", "")
base_dir = f"{WORK}/physics-datasets"

In [ ]:
DEFAULT_COLORS = list(plt.rcParams["axes.prop_cycle"].by_key()["color"])

In [ ]:
metrics = ["mse", "mae", "smape"]

## Load Model Checkpoints

In [ ]:
# run_name = "pft_fourier_noemb_fromscratch-0"
run_name = "pft_chattn_emb_w_poly-0"  # NOTE: this is still the best
# run_name = "pft_chattn_noembed_pretrained_correct-0"  # chattn + mlm
# run_name = "pft_linattnpolyemb_from_scratch-0"

pft_model = PatchTSTPipeline.from_pretrained(
    mode="predict",
    pretrain_path=f"/stor/work/AMDG_Gilpin_Summer2024/checkpoints/{run_name}/checkpoint-final",
    device_map="cuda:2",
)

In [ ]:
run_name = "chronos_mini_ft-0"
chronos_ft = ChronosPipeline.from_pretrained(
    f"/stor/work/AMDG_Gilpin_Summer2024/checkpoints/{run_name}-0/checkpoint-final",
    device_map="cuda:3",
    torch_dtype=torch.float32,
)

chronos_ft_kwargs = {
    "transpose": True,
    "limit_prediction_length": False,
    "num_samples": 1,
    "deterministic": True,
}

In [ ]:
run_name = "amazon/chronos-t5-mini"
chronos_zs = ChronosPipeline.from_pretrained(
    run_name, device_map="cuda:4", torch_dtype=torch.float32
)

chronos_zs_kwargs = {
    "transpose": True,
    "limit_prediction_length": False,
    "num_samples": 1,
    "deterministic": True,
}

## Forecast and Plot Utils

In [ ]:
def forecast(
    model,
    context: np.ndarray,
    prediction_length: int,
    transpose: bool = False,
    standardize: bool = True,
    differenced: bool = False,
    **kwargs,
) -> np.ndarray:
    """
    Args:
        model: The model to use for forecasting.
        context: The context to forecast (n_timesteps, n_features)
        context_length: The length of the context.
        prediction_length: The length of the prediction.
        transpose: Whether to transpose the data.

    Returns:
        The forecasted data (prediction_length, n_features)
    """
    preprocessed_context = context.copy()

    if differenced:
        differenced_context = np.diff(preprocessed_context, axis=0)
        preprocessed_context = differenced_context.copy()
    if standardize:
        preprocessed_context = safe_standardize(preprocessed_context, axis=0)

    context_tensor = torch.from_numpy(
        preprocessed_context.T if transpose else preprocessed_context
    ).float()
    pred = (
        model.predict(context_tensor, prediction_length, verbose=False, **kwargs)
        .squeeze()
        .cpu()
        .numpy()
    )
    if transpose:
        pred = pred.T

    if standardize:
        pred = safe_standardize(
            pred,
            axis=0,
            context=differenced_context if differenced else context,
            denormalize=True,
        )
    if differenced:
        pred = np.cumsum(pred, axis=0) + context[-1]

    # prediction length may be shorter than model output length
    return pred[:prediction_length, :] if pred.ndim == 2 else pred[:prediction_length]


def compute_rollout_metrics(
    model,
    data: np.ndarray,
    context_length: int,
    prediction_length: int,
    starts: np.ndarray | list[int] | None = None,
    num_windows: int | None = None,
    step: int = 64,
    metrics: list[str] = ["mse", "mae", "smape"],
    **kwargs,
) -> tuple[
    dict[str, np.ndarray],
    dict[str, np.ndarray],
    np.ndarray | list[int],
    list[np.ndarray],
]:
    if starts is not None:
        assert num_windows is None, "num_windows must be None if starts is provided"
        num_windows = len(starts)
    else:
        if num_windows is None:
            raise ValueError("num_windows must be provided if starts is not provided")
        starts = np.random.randint(
            0, len(data) - context_length - prediction_length, num_windows
        )

    assert len(starts) == num_windows, "starts must be a list of length num_windows"
    assert max(starts) < len(data) - context_length - prediction_length, (
        "starts must be less than the length of the data"
    )

    full_metrics = defaultdict(
        lambda: np.zeros((num_windows, prediction_length // step))
    )

    predictions = []
    for s in tqdm(range(num_windows), desc="Sampling contexts", total=num_windows):
        start = starts[s]
        context = data[start : start + context_length]
        prediction = forecast(model, context, prediction_length, **kwargs)
        for i in range(0, prediction_length, step):
            pred = prediction[i : i + step]

            gt = data[start + context_length + i : start + context_length + i + step]
            submetrics = compute_metrics(pred, gt, include=metrics)
            for k, v in submetrics.items():
                full_metrics[k][s, i // step] += v
        predictions.append(prediction)
    mean_metrics = {k: v.mean(axis=0) for k, v in full_metrics.items()}
    std_metrics = {
        k: v.std(axis=0) / np.sqrt(num_windows) for k, v in full_metrics.items()
    }
    return mean_metrics, std_metrics, starts, predictions


def plot_model_prediction(
    model,
    data: np.ndarray,
    context_length: int,
    prediction_length: int,
    transpose: bool = False,
    standardize: bool = True,
    save_path: str | None = None,
    color: str = "red",
    **kwargs,
):
    context = data[:context_length]
    groundtruth = data[context_length : context_length + prediction_length]
    prediction = forecast(
        model, context, prediction_length, transpose, standardize, **kwargs
    )

    total_length = context_length + prediction_length
    context_ts = np.arange(context_length + 1)
    pred_ts = np.arange(context_length, total_length)

    fig = plt.figure(figsize=(15, 4))
    outer_grid = fig.add_gridspec(1, 2, width_ratios=[0.5, 0.5], wspace=0.05)
    gs = outer_grid[1].subgridspec(3, 1, height_ratios=[1 / 3] * 3, wspace=0, hspace=0)
    ax_3d = fig.add_subplot(outer_grid[0], projection="3d")
    ax_3d.plot(*context.T[:3], alpha=0.5, color="black", label="Context")
    ax_3d.plot(*groundtruth.T[:3], linestyle="-", color="black", label="Groundtruth")
    ax_3d.plot(*prediction.T[:3], color=color, label="Prediction")
    ax_3d.legend(loc="upper right", fontsize=8)
    ax_3d.set_xlabel("$x_1$")
    ax_3d.set_ylabel("$x_2$")
    ax_3d.set_zlabel("$x_3$")

    # Make clean projection
    ax_3d.grid(False)
    ax_3d.set_facecolor("white")
    ax_3d.set_xticks([])
    ax_3d.set_yticks([])
    ax_3d.set_zticks([])
    ax_3d.axis("off")

    axes_1d = [fig.add_subplot(gs[i, 0]) for i in range(3)]
    for i, ax in enumerate(axes_1d):
        ax.plot(
            context_ts,
            data[: context_length + 1, i],
            alpha=0.5,
            color="black",
        )
        ax.plot(pred_ts, groundtruth[:, i], linestyle="-", color="black")
        ax.plot(pred_ts, prediction[:, i], color=color)
        ax.set_ylabel(f"$x_{i + 1}$")
        ax.set_aspect("auto")
    axes_1d[-1].set_xlabel("Time")

    if save_path is None:
        plt.show()
    else:
        plt.savefig(save_path)
    plt.close()

In [ ]:
def plot_forecast_3d(
    data: np.ndarray,
    predictions_dict: dict[str, np.ndarray],
    context_length: int,
    prediction_length: int,
    figsize: tuple[int, int] = (6, 6),
    show_legend: bool = True,
    legend_kwargs: dict[str, Any] = {},
    save_path: str | None = None,
):
    context = data[: context_length + 1, :3]
    groundtruth = data[context_length : context_length + prediction_length, :3]

    plt.figure(figsize=figsize)
    ax = plt.axes(projection="3d")
    ax._axis3don = False

    # Combine all data to find min/max bounds
    all_data = [context, groundtruth] + [
        pred[:, :3] for pred in predictions_dict.values()
    ]
    mins = np.array([d.min(axis=0) for d in all_data])
    maxs = np.array([d.max(axis=0) for d in all_data])

    xmin, ymin, zmin = np.min(mins, axis=0)
    xmax, ymax, zmax = np.max(maxs, axis=0)

    ax.xaxis.pane.set_visible(False)
    ax.yaxis.pane.set_visible(False)
    ax.zaxis.pane.set_visible(False)
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.plot3D(*context.T, alpha=0.1, color="black", zorder=1)
    ax.plot3D(
        *groundtruth.T,
        alpha=0.8,
        color="black",
        linestyle="-",
        zorder=2,
        label="Ground Truth",
    )
    for model_name, prediction in predictions_dict.items():
        ax.plot3D(
            *prediction[:, :3].T,
            label=model_name,
            zorder=10 if model_name == "Our Model" else 1,
        )
    if show_legend:
        ax.legend(**legend_kwargs)

    ax.quiver(
        xmin,
        ymax,
        zmin,
        xmax - xmin,
        0,
        0,
        color="black",
        arrow_length_ratio=0.05,
        zorder=5,
    )
    ax.quiver(
        xmin,
        ymax,
        zmin,
        0,
        -ymax + ymin,
        0,
        color="black",
        arrow_length_ratio=0.05,
        zorder=5,
    )
    ax.quiver(
        xmin,
        ymax,
        zmin,
        0,
        0,
        zmax - zmin,
        color="black",
        arrow_length_ratio=0.05,
        zorder=5,
    )

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

In [ ]:
def plot_metric_comparison(
    model_metrics: dict[str, tuple[dict[str, np.ndarray], dict[str, np.ndarray]]],
    prediction_length: int,
    compute_metrics_time_interval: int,
    metric_name: str = "smape",
    colors: list[str] = DEFAULT_COLORS,
    markers: str = DEFAULT_MARKERS,
    title: str | None = None,
    figsize: tuple[float, float] = (4, 3),
    show_legend: bool = True,
    legend_kwargs: dict[str, Any] = {},
    ylim: tuple[float | None, float | None] | None = None,
    save_path: str | None = None,
    metric_name_mapping: dict[str, str] = {"smape": "sMAPE"},
):
    """
    Plot comparison between different models on a given metric.

    Args:
        model_metrics: Dictionary with model names as keys and tuples of (mean_metrics, std_metrics) as values
        metric_name: Name of the metric to plot
        prediction_length: Length of prediction
        compute_metrics_time_interval: Time interval for computing metrics
        save_path: Path to save the figure
    """
    plt.figure(figsize=figsize)
    ts = np.arange(
        compute_metrics_time_interval,
        prediction_length + compute_metrics_time_interval,
        compute_metrics_time_interval,
    )

    for i, (model_name, (mean_metrics, std_metrics)) in enumerate(
        model_metrics.items()
    ):
        plt.plot(
            ts,
            mean_metrics[metric_name],
            color=colors[i],
            marker=markers[i],
            label=model_name,
        )
        plt.fill_between(
            ts,
            mean_metrics[metric_name] - std_metrics[metric_name],
            mean_metrics[metric_name] + std_metrics[metric_name],
            alpha=0.1,
            color=colors[i],
        )

    metric_name_title = metric_name.upper()
    if metric_name in metric_name_mapping:
        metric_name_title = metric_name_mapping[metric_name]

    plt.ylabel(metric_name_title, fontweight="bold")
    plt.xlabel("Prediction Length", fontweight="bold")
    if show_legend:
        plt.legend(loc="lower right", frameon=True, **legend_kwargs)
    plt.xticks(ts)
    plt.tight_layout()
    if title is not None:
        plt.title(title, fontweight="bold")
    if ylim is not None:
        plt.ylim(*ylim)
    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

# Turbulent Boundary Layer

In [ ]:
turbpca_data = np.load(
    f"{base_dir}/turbulence/BLexp_Re980_pca10.pkl", allow_pickle=True
)
transient = 1024
subsampled_turbpca_data = turbpca_data[transient::2]
print(subsampled_turbpca_data.shape)

In [ ]:
context_length = 512
prediction_length = 512

differenced = False

In [ ]:
pft_prediction = forecast(
    pft_model,
    subsampled_turbpca_data[:context_length],
    prediction_length,
    limit_prediction_length=False,
    sliding_context=False,
    differenced=differenced,
)

In [ ]:
chronos_ft_prediction = forecast(
    chronos_ft,
    subsampled_turbpca_data[:context_length],
    prediction_length,
    differenced=differenced,
    **chronos_ft_kwargs,
)

In [ ]:
chronos_zs_prediction = forecast(
    chronos_zs,
    subsampled_turbpca_data[:context_length],
    prediction_length,
    differenced=differenced,
    **chronos_zs_kwargs,
)

In [ ]:
plot_forecast_3d(
    turbpca_data,
    {
        "Our Model": pft_prediction,
        "Chronos 20M SFT": chronos_ft_prediction,
        "Chronos 20M": chronos_zs_prediction,
    },
    context_length,
    prediction_length,
    show_legend=False,
    legend_kwargs={"loc": "center right", "frameon": True},
    save_path="../figures/turbpca_comparison.pdf",
)

In [ ]:
# sanity check
_ = plot_model_prediction(
    pft_model,
    subsampled_turbpca_data,
    context_length,
    prediction_length,
    sliding_context=True,
    limit_prediction_length=False,
    differenced=differenced,
    color=DEFAULT_COLORS[0],
    # save_path="../figures/turbpca_comparison_ourmodel.pdf",
)
_ = plot_model_prediction(
    chronos_ft,
    subsampled_turbpca_data,
    context_length,
    prediction_length,
    differenced=differenced,
    color=DEFAULT_COLORS[1],
    **chronos_ft_kwargs,
    # save_path="../figures/turbpca_comparison_chronosft.pdf",
)
_ = plot_model_prediction(
    chronos_zs,
    subsampled_turbpca_data,
    context_length,
    prediction_length,
    differenced=differenced,
    **chronos_zs_kwargs,
    color=DEFAULT_COLORS[2],
    # save_path="../figures/turbpca_comparison_chronoszs.pdf",
)

In [ ]:
compute_metrics_time_interval = 64
turbpca_start_times = np.arange(
    0, len(subsampled_turbpca_data) - context_length - prediction_length, 256
)
print(f"Number of windows: {len(turbpca_start_times)}")

In [ ]:
pft_mean_metrics, pft_std_metrics, _, pft_predictions = compute_rollout_metrics(
    pft_model,
    subsampled_turbpca_data,
    context_length,
    prediction_length,
    starts=turbpca_start_times,
    step=compute_metrics_time_interval,
    sliding_context=True,
    limit_prediction_length=False,
    differenced=differenced,
)

In [ ]:
chronos_ft_mean_metrics, chronos_ft_std_metrics, _, chronos_ft_predictions = (
    compute_rollout_metrics(
        chronos_ft,
        subsampled_turbpca_data,
        context_length,
        prediction_length,
        starts=turbpca_start_times,
        step=compute_metrics_time_interval,
        differenced=differenced,
        **chronos_ft_kwargs,
    )
)

In [ ]:
chronos_zs_mean_metrics, chronos_zs_std_metrics, _, chronos_zs_predictions = (
    compute_rollout_metrics(
        chronos_zs,
        subsampled_turbpca_data,
        context_length,
        prediction_length,
        starts=turbpca_start_times,
        step=compute_metrics_time_interval,
        differenced=differenced,
        **chronos_zs_kwargs,
    )
)

In [ ]:
model_metrics = {
    "Our Model": (pft_mean_metrics, pft_std_metrics),
    "Chronos 20M SFT": (chronos_ft_mean_metrics, chronos_ft_std_metrics),
    "Chronos 20M": (chronos_zs_mean_metrics, chronos_zs_std_metrics),
}

plot_metric_comparison(
    model_metrics,
    prediction_length,
    compute_metrics_time_interval,
    metric_name="smape",
    save_path="../figures/turbpca_comparison_smape.pdf",
)

# ECG

In [ ]:
fpath = f"{base_dir}/electrocardiogram/ecg_train.csv.gz"
ecg_data = np.loadtxt(fpath, delimiter=",")
print(ecg_data.shape)

In [ ]:
context_length = 512
prediction_length = 512
start = 0
stride = 1
differenced = False

subsampled_ecg_data = ecg_data[start::stride]

start_times_to_show = np.arange(0, len(subsampled_ecg_data) - 1024, 20000)
start_times_extra = np.arange(4096, len(subsampled_ecg_data) - 1024, 4096)
start_times = np.concatenate([start_times_to_show, start_times_extra])
compute_metrics_times = np.arange(64, prediction_length + 64, 64)

In [ ]:
our_rollout_metrics = defaultdict(dict)
our_predictions = defaultdict()

for start_time in tqdm(
    start_times, desc=f"Forecasting PFT for {len(start_times)} context windows..."
):
    pft_prediction = forecast(
        pft_model,
        subsampled_ecg_data[start_time : start_time + context_length],
        prediction_length,
        limit_prediction_length=False,
        sliding_context=True,
        differenced=differenced,
    )
    our_predictions[start_time] = pft_prediction
    for t in compute_metrics_times:
        submetrics = compute_metrics(
            pft_prediction[0:t],
            subsampled_ecg_data[context_length : context_length + t],
            include=metrics,
        )
        for metric_name in metrics:
            if t not in our_rollout_metrics[metric_name]:
                our_rollout_metrics[metric_name][t] = []
            our_rollout_metrics[metric_name][t].append(submetrics[metric_name])
print(our_rollout_metrics["smape"])

In [ ]:
chronos_ft_rollout_metrics = defaultdict(dict)
chronos_ft_predictions = defaultdict()

for start_time in tqdm(
    start_times,
    desc=f"Forecasting Chronos FT for {len(start_times)} context windows...",
):
    chronos_prediction = forecast(
        chronos_ft,
        subsampled_ecg_data[start_time : start_time + context_length],
        prediction_length,
        **chronos_ft_kwargs,
        differenced=differenced,
    )
    chronos_ft_predictions[start_time] = chronos_prediction
    for t in compute_metrics_times:
        submetrics = compute_metrics(
            chronos_prediction[0:t],
            subsampled_ecg_data[context_length : context_length + t],
            include=metrics,
        )
        for metric_name in metrics:
            if t not in chronos_ft_rollout_metrics[metric_name]:
                chronos_ft_rollout_metrics[metric_name][t] = []
            chronos_ft_rollout_metrics[metric_name][t].append(submetrics[metric_name])
print(chronos_ft_rollout_metrics["smape"])

In [ ]:
chronos_zs_rollout_metrics = defaultdict(dict)
chronos_zs_predictions = defaultdict()

for start_time in tqdm(
    start_times,
    desc=f"Forecasting Chronos ZS for {len(start_times)} context windows...",
):
    chronos_prediction = forecast(
        chronos_zs,
        subsampled_ecg_data[start_time : start_time + context_length],
        prediction_length,
        **chronos_zs_kwargs,
        differenced=differenced,
    )
    chronos_zs_predictions[start_time] = chronos_prediction
    for t in compute_metrics_times:
        submetrics = compute_metrics(
            chronos_prediction[0:t],
            subsampled_ecg_data[context_length : context_length + t],
            include=metrics,
        )
        for metric_name in metrics:
            if t not in chronos_zs_rollout_metrics[metric_name]:
                chronos_zs_rollout_metrics[metric_name][t] = []
            chronos_zs_rollout_metrics[metric_name][t].append(submetrics[metric_name])
print(chronos_zs_rollout_metrics["smape"])

In [ ]:
prediction_length_to_show = 512
step = 1

context_ts = np.arange(context_length + 1)
pred_ts = np.arange(context_length, context_length + prediction_length_to_show)
error_ts = (
    np.arange(context_length - step, context_length + prediction_length_to_show, step)
    + step
)

selected_start_times = [4096, 40_000, 60_000]
n_examples = len(selected_start_times)

fig, axes = plt.subplots(n_examples, 1, figsize=(4, 3), sharex=True)
plt.subplots_adjust(hspace=0.0)

linewidth = 1
for i, start_time in enumerate(selected_start_times):
    axes[i].plot(
        context_ts,
        subsampled_ecg_data[start_time : start_time + context_length + 1],
        color="black",
        alpha=0.2,
        linewidth=linewidth,
        label="context",
    )
    axes[i].plot(
        pred_ts,
        subsampled_ecg_data[
            start_time + context_length : start_time
            + context_length
            + prediction_length_to_show
        ],
        color="black",
        linestyle="-",
        linewidth=linewidth,
        label="groundtruth",
        alpha=0.8,
    )
    axes[i].plot(
        pred_ts,
        our_predictions[start_time][:prediction_length_to_show],
        # color=colors[0],
        linewidth=linewidth,
        zorder=10,
        # alpha=0.8,
    )
    axes[i].plot(
        pred_ts,
        chronos_ft_predictions[start_time][:prediction_length_to_show],
        # color=colors[1],
        linewidth=linewidth,
        alpha=0.8,
        zorder=2,
    )
    axes[i].plot(
        pred_ts,
        chronos_zs_predictions[start_time][:prediction_length_to_show],
        # color=colors[2],
        linewidth=linewidth if i < 2 else 2,
        alpha=0.8 if i < 2 else 1,
        zorder=1 if i < 2 else 8,
    )

    axes[i].set_xticks([])
    axes[i].set_yticks([])

plt.tight_layout()
plt.savefig("../figures/ecg_all_models_forecasts.pdf", bbox_inches="tight")
plt.show()

In [ ]:
def calculate_stats(data_dict: dict[int, np.ndarray | list[float]]):
    """
    Calculate mean, standard deviation, and standard error for each model
    """
    mean_vals = {t: np.mean(v) for t, v in data_dict.items()}
    median_vals = {t: np.median(v) for t, v in data_dict.items()}
    std_vals = {t: np.std(v) for t, v in data_dict.items()}
    ste_vals = {t: std_vals[t] / np.sqrt(len(data_dict[t])) for t in data_dict.keys()}
    return mean_vals, median_vals, std_vals, ste_vals


def plot_model_results(
    mean_dict: dict[int, float],
    ste_dict: dict[int, float],
    marker: str,
    label: str,
    color: str | None = None,
):
    """
    Helper function to plot a model's results with error bands
    """
    x_values = list(mean_dict.keys())
    y_values = list(mean_dict.values())
    y_errors = list(ste_dict.values())

    plt.plot(x_values, y_values, marker=marker, label=label, color=color)
    plt.fill_between(
        x_values,
        np.array(y_values) - np.array(y_errors),
        np.array(y_values) + np.array(y_errors),
        alpha=0.1,
    )


metric_name = "smape"
metric_name_title = "sMAPE"
# Calculate statistics for all models
mean_metric_ours, median_metric_ours, _, ste_metric_ours = calculate_stats(
    our_rollout_metrics[metric_name]
)
mean_metric_chronos_ft, median_metric_chronos_ft, _, ste_metric_chronos_ft = (
    calculate_stats(chronos_ft_rollout_metrics[metric_name])
)
mean_metric_chronos_zs, median_metric_chronos_zs, _, ste_metric_chronos_zs = (
    calculate_stats(chronos_zs_rollout_metrics[metric_name])
)


# Create the plot
plt.figure(figsize=(4, 3))

# Plot each model
plot_model_results(mean_metric_ours, ste_metric_ours, "o", label="Our Model")
plot_model_results(
    mean_metric_chronos_ft, ste_metric_chronos_ft, "s", label="Chronos 20M SFT"
)
plot_model_results(
    mean_metric_chronos_zs, ste_metric_chronos_zs, "v", label="Chronos ZS"
)

plt.xlabel("Prediction Length", fontweight="bold")
plt.xticks(list(mean_metric_ours.keys()))
plt.ylabel(metric_name_title, fontweight="bold")

plt.legend(loc="lower center", frameon=True, ncol=2)
plt.tight_layout()
plt.savefig(f"../figures/ecg_all_models_{metric_name}.pdf", bbox_inches="tight")
plt.show()